# JevLite on Colab

Train a typed decision engine that answers arbitrary typed questions
about a state in **one forward pass**, and score it against
[TypeSafe Jev](https://www.mindstudio.ai/blog/jev-system-one-model-launch)
on the public
[`LocalLLaMA/typed-decisions`](https://huggingface.co/datasets/LocalLLaMA/typed-decisions)
test split.

| | accuracy |
|---|---|
| majority-class baseline | 0.483 (measured) |
| JevLite, 6 epochs | 0.568 (measured - undertrained) |
| single random annotator | 0.659 (measured) |
| TypeSafe Jev 1.13.0 | 0.727 (published) |
| best open result reported | 0.766 |

The 6-epoch run stopped while validation accuracy was still climbing
steeply, so this notebook now trains for 20 with early stopping. On
calibration and speed that same run already beat Jev: **ECE 0.065 vs
0.144** and **65 ms/case on a T4 vs 239 ms** over Jev's API.

**No API key is needed anywhere in this notebook.** Total runtime on a
free T4 is roughly 20 minutes.

Set the runtime first: **Runtime -> Change runtime type -> T4 GPU**.

## 0. Environment

In [ ]:
!nvidia-smi -L || echo 'NO GPU - set Runtime > Change runtime type > T4 GPU'

In [ ]:
%pip install -q 'transformers>=4.48' 'datasets>=2.20' scikit-learn

In [ ]:
# Two things `!cmd` gets wrong in a notebook: it does NOT stop `Run all`
# when a command fails (the error surfaces cells later as a confusing
# FileNotFoundError), and output routed through a pipe is block-buffered,
# so a long job looks frozen. This streams line by line AND raises.
import os
import subprocess

def run(cmd):
    print('$', cmd, flush=True)
    env = dict(os.environ, PYTHONUNBUFFERED='1')
    p = subprocess.Popen(cmd, shell=True, env=env, text=True,
                         bufsize=1, stdout=subprocess.PIPE,
                         stderr=subprocess.STDOUT)
    for line in p.stdout:
        print(line, end='', flush=True)
    if p.wait():
        print()
        raise SystemExit(
            f'*** FAILED (exit {p.returncode}): {cmd} -- fix this '
            'before running the cells below, they depend on it.')
    print('ok', flush=True)

## 1. Write the repo

Everything below runs the same scripts you would run locally, so any
result here reproduces off Colab unchanged.

In [ ]:
# Writes the repo into the Colab filesystem. Generated by
# make_notebook.py - edit the .py files and regenerate, not this cell.
import pathlib

FILES = {}

FILES['typed_schema.py'] = r'''
"""The Jev interface, reimplemented.

A *state* (any JSON or text) plus N *typed questions*. One forward pass answers
all of them. Three primitive types, matching Jev:

  noul   - boolean          criteria: {"true": desc, "false": desc}
  choice - enum             criteria: {label: desc, ...}
  score  - ordered scale    criteria: [desc_0, desc_1, ...]   labels are "0".."n-1"

Everything downstream reads questions through `iter_labels`, so adding a new
primitive means touching this file only.
"""
from __future__ import annotations
import json
from typing import Any

TYPES = ["noul", "choice", "score"]
TYPE_ID = {t: i for i, t in enumerate(TYPES)}


# A bare noul carries no criteria at all - the instruction is the statement and
# the labels are implicit. 10% of typed-decisions is shaped this way, so treat it
# as part of the format rather than as malformed input.
BARE_NOUL = [("false", "The statement does not hold."),
             ("true", "The statement holds.")]


def iter_labels(q: dict) -> list[tuple[str, str]]:
    """Normalize a question's criteria to an ordered [(label, description)] list.

    `score` criteria arrive as an ordered list; its labels are the indices as
    strings, which is how the gold distributions key them.
    """
    crit = q.get("criteria")
    if crit is None:
        if q["type"] != "noul":
            raise ValueError(
                f"a {q['type']} question needs criteria - only noul may omit "
                f"them: {q.get('instructions', '')[:80]!r}")
        return list(BARE_NOUL)
    if isinstance(crit, list):                      # score
        return [(str(i), d) for i, d in enumerate(crit)]
    if q["type"] == "noul":                         # pin order: false, true
        return [(k, crit[k]) for k in ("false", "true") if k in crit]
    return sorted(crit.items())                     # choice - stable order


def state_to_text(state: Any) -> str:
    if isinstance(state, str):
        return state
    return json.dumps(state, indent=1, sort_keys=True, ensure_ascii=False)


def label_text(label: str, desc: str) -> str:
    return f"{label}: {desc}"


class Question(dict):
    """Convenience builder so you can define a schema in code.

    >>> Question.choice("What should we do?", {"retry": "...", "stop": "..."})
    """

    @staticmethod
    def noul(instructions: str, true_desc: str = None,
             false_desc: str = None) -> dict:
        """Descriptions are optional, matching the data: a bare noul carries no
        criteria at all and the instruction is the statement being judged."""
        q = {"type": "noul", "instructions": instructions}
        if true_desc is not None or false_desc is not None:
            q["criteria"] = {"true": true_desc or "The statement holds.",
                             "false": false_desc or "The statement does not hold."}
        return q

    @staticmethod
    def choice(instructions: str, criteria: dict[str, str]) -> dict:
        return {"type": "choice", "instructions": instructions, "criteria": criteria}

    @staticmethod
    def score(instructions: str, levels: list[str]) -> dict:
        return {"type": "score", "instructions": instructions, "criteria": levels}
'''

FILES['td_data.py'] = r'''
"""Load LocalLLaMA/typed-decisions and turn it into label-position tensors.

One example = one state + every question about it. The encoder sees all of them
in a single sequence; each candidate label gets a marker token whose hidden state
becomes that label's logit.
"""
from __future__ import annotations
import hashlib
import json
import torch
from torch.utils.data import Dataset
from typed_schema import iter_labels, state_to_text, label_text, TYPE_ID

DATASET = "LocalLLaMA/typed-decisions"
Q_TOKEN, L_TOKEN = "<<q>>", "<<l>>"
MAX_Q = 8


def _parse(r):
    state = r["state"]
    return {
        "id": r["id"],
        "workflow": r["workflow"],
        "state": json.loads(state) if state.lstrip()[:1] in "{[" else state,
        "questions": json.loads(r["questions"]),
        "gold": json.loads(r["gold"]) if r.get("gold") else {},
    }


def _load_http(split, config, limit):
    """Fallback for environments without the `datasets` package (and for Colab
    cold starts, where the rows endpoint is faster than a parquet download)."""
    import time, urllib.error, urllib.parse, urllib.request

    def get(url, tries=5):
        # The anonymous rows endpoint rate-limits, and it does so exactly when
        # you are iterating quickly. Back off rather than dying mid-download.
        for i in range(tries):
            try:
                with urllib.request.urlopen(url, timeout=60) as f:
                    return json.load(f)
            except urllib.error.HTTPError as e:
                if e.code not in (429, 502, 503, 504) or i == tries - 1:
                    raise
                wait = 2 ** i * 5
                print(f"  HTTP {e.code} from the dataset server, retrying in {wait}s "
                      f"({i+1}/{tries-1})")
                time.sleep(wait)

    rows, offset = [], 0
    while limit is None or len(rows) < limit:
        n = 100 if limit is None else min(100, limit - len(rows))
        q = urllib.parse.urlencode({"dataset": DATASET, "config": config,
                                    "split": split, "offset": offset, "length": n})
        page = get(f"https://datasets-server.huggingface.co/rows?{q}")
        got = [_parse(x["row"]) for x in page["rows"]]
        rows += got
        offset += len(got)
        if not got or offset >= page.get("num_rows_total", offset):
            break
    return rows[:limit] if limit else rows


def load_split(split: str, config: str = "all", limit: int | None = None):
    try:
        from datasets import load_dataset
    except ImportError:
        return _load_http(split, config, limit)
    ds = load_dataset(DATASET, config, split=split)
    rows = []
    for r in ds:
        rows.append(_parse(r))
        if limit and len(rows) >= limit:
            break
    return rows


def add_markers(tok):
    """Register the two marker tokens. Caller must resize the model embeddings."""
    n = tok.add_tokens([Q_TOKEN, L_TOKEN], special_tokens=True)
    return n, tok.convert_tokens_to_ids(Q_TOKEN), tok.convert_tokens_to_ids(L_TOKEN)


def encode(row, tok, max_len, qid, lid, with_gold=True):
    """-> dict of python lists. Labels are laid out question by question."""
    qnames = sorted(row["questions"])[:MAX_Q]

    tail, label_pos, group, target, qtypes = [], [], [], [], []
    for gi, qname in enumerate(qnames):
        q = row["questions"][qname]
        qtypes.append(TYPE_ID[q["type"]])
        tail += [qid] + tok.encode(q["instructions"], add_special_tokens=False)
        gold = row.get("gold", {}).get(qname, {}) if with_gold else {}
        probs = gold.get("probabilities", {})
        for lab, desc in iter_labels(q):
            label_pos.append(len(tail))            # index of the <<l>> marker
            group.append(gi)
            target.append(float(probs.get(lab, 0.0)))
            tail += [lid] + tok.encode(label_text(lab, desc), add_special_tokens=False)

    # The questions are the part we must never truncate; the state absorbs the cut.
    budget = max_len - len(tail) - 2
    if budget < 32:
        raise ValueError(f"max_len={max_len} too small for {len(tail)} question tokens")
    head = [tok.cls_token_id] + tok.encode(
        state_to_text(row["state"]), add_special_tokens=False)[:budget]

    off = len(head)
    ids = head + tail + [tok.sep_token_id]
    return {
        "input_ids": ids,
        "label_pos": [p + off for p in label_pos],
        "group": group,
        "target": target,
        "qtype": qtypes,
        "qnames": qnames,
        # .get chain: rows built for inference carry no gold at all, and
        # predict_distributions used to die on them with a bare KeyError.
        "gold_label": [row.get("gold", {}).get(q, {}).get("label") for q in qnames],
        "labels": [[l for l, _ in iter_labels(row["questions"][q])] for q in qnames],
    }


class TypedDecisions(Dataset):
    def __init__(self, rows, tok, max_len, qid, lid):
        self.enc = [encode(r, tok, max_len, qid, lid) for r in rows]
        self.rows = rows

    def __len__(self):
        return len(self.enc)

    def __getitem__(self, i):
        return self.enc[i]


def collate(batch, pad_id):
    B = len(batch)
    L = max(len(b["input_ids"]) for b in batch)
    M = max(len(b["label_pos"]) for b in batch)
    Q = max(len(b["qtype"]) for b in batch)

    ids = torch.full((B, L), pad_id, dtype=torch.long)
    att = torch.zeros((B, L), dtype=torch.long)
    pos = torch.zeros((B, M), dtype=torch.long)
    grp = torch.full((B, M), -1, dtype=torch.long)
    tgt = torch.zeros((B, M), dtype=torch.float)
    lmask = torch.zeros((B, M), dtype=torch.bool)
    qtype = torch.full((B, Q), -1, dtype=torch.long)
    qmask = torch.zeros((B, Q), dtype=torch.bool)

    for i, b in enumerate(batch):
        n, m, q = len(b["input_ids"]), len(b["label_pos"]), len(b["qtype"])
        ids[i, :n] = torch.tensor(b["input_ids"])
        att[i, :n] = 1
        pos[i, :m] = torch.tensor(b["label_pos"])
        grp[i, :m] = torch.tensor(b["group"])
        tgt[i, :m] = torch.tensor(b["target"])
        lmask[i, :m] = True
        qtype[i, :q] = torch.tensor(b["qtype"])
        qmask[i, :q] = True

    return {"input_ids": ids, "attention_mask": att, "label_pos": pos,
            "group": grp, "target": tgt, "label_mask": lmask,
            "qtype": qtype, "qmask": qmask}


def val_split(rows, frac=0.15):
    """Deterministic train/val carve-out of the 1200-case train split.

    Hash-based on id so the same cases stay in validation when you add data or
    rerun on another machine - temperature fitted on a moving val set is not a
    calibration, it is a lottery.
    """
    def h(r):
        return int(hashlib.sha1(r["id"].encode()).hexdigest()[:8], 16) / 0xFFFFFFFF
    return [r for r in rows if h(r) >= frac], [r for r in rows if h(r) < frac]


def pad_cat(tensors, pad_value=0):
    """Concatenate [B, M] tensors whose M differs, padding to the widest.

    Batches are collated independently, so their label dimension is only as
    wide as their own widest example. A plain torch.cat across batches works
    right up until two batches contain different workflows - which is why this
    only shows up on the full dataset and never on a single-workflow slice.
    """
    width = max(t.size(1) for t in tensors)
    out = []
    for t in tensors:
        if t.size(1) < width:
            pad = torch.full((t.size(0), width - t.size(1)), pad_value,
                             dtype=t.dtype, device=t.device)
            t = torch.cat([t, pad], dim=1)
        out.append(t)
    return torch.cat(out, dim=0)
'''

FILES['model.py'] = r'''
"""JevLite: one encoder pass, a logit per candidate label.

The schema is data, not architecture. Every candidate label is written into the
input sequence behind a marker token; the head is a single Linear(d, 1) applied
at each marker position. Adding a question, a label, or a whole new workflow at
inference time requires no retraining and no new parameters.

Compare to a fixed-head classifier, which needs one nn.Linear per question and
can only ever answer the questions it was built with.
"""
from __future__ import annotations
import os
import sys
import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import AutoModel, AutoTokenizer
from td_data import add_markers
from typed_schema import TYPES

DEFAULT_ENCODER = "answerdotai/ModernBERT-base"   # bidirectional, 8192 ctx, 150M
NEG = -1e4                                        # fp16-safe masking value


def grouped_softmax(logits, group, label_mask, n_groups):
    """Softmax within each question's own label set. -> probs [B, M]"""
    probs = torch.zeros_like(logits)
    for q in range(n_groups):
        m = (group == q) & label_mask
        if not m.any():
            continue
        masked = logits.masked_fill(~m, NEG)
        p = torch.softmax(masked, dim=-1)
        probs = probs + p * m
    return probs


class JevLite(nn.Module):
    def __init__(self, encoder_name=DEFAULT_ENCODER, tokenizer=None, dropout=0.1):
        super().__init__()
        self.encoder_name = encoder_name
        self.tok = tokenizer or AutoTokenizer.from_pretrained(encoder_name)
        _, self.qid, self.lid = add_markers(self.tok)
        self.encoder = AutoModel.from_pretrained(encoder_name)
        self.encoder.resize_token_embeddings(len(self.tok))
        d = self.encoder.config.hidden_size
        self.drop = nn.Dropout(dropout)
        self.score = nn.Linear(d, 1)
        # One temperature per primitive type. Jev's measured failure is that its
        # types miscalibrate in opposite directions, so a single global T cannot
        # fix both. These are frozen during training and fitted in 03_calibrate.py.
        self.log_temp = nn.Parameter(torch.zeros(len(TYPES)), requires_grad=False)

    def logits(self, input_ids, attention_mask, label_pos):
        h = self.encoder(input_ids=input_ids,
                         attention_mask=attention_mask).last_hidden_state
        d = h.size(-1)
        picked = h.gather(1, label_pos.unsqueeze(-1).expand(-1, -1, d))
        return self.score(self.drop(picked)).squeeze(-1)          # [B, M]

    def forward(self, batch, apply_temperature=False):
        lg = self.logits(batch["input_ids"], batch["attention_mask"], batch["label_pos"])
        if apply_temperature:
            lg = lg / self.temp_per_label(batch)
        n_groups = batch["qtype"].size(1)
        probs = grouped_softmax(lg.float(), batch["group"], batch["label_mask"], n_groups)
        return lg, probs

    def temp_per_label(self, batch):
        """Map each label slot to its question's temperature. -> [B, M]"""
        g = batch["group"].clamp(min=0)
        t = batch["qtype"].clamp(min=0).gather(1, g)              # type id per label
        return self.log_temp.exp()[t]


def decision_loss(probs, batch, brier_weight=1.0):
    """Soft cross-entropy against the consensus distribution, plus Brier.

    The dataset gives full annotator distributions, so training on the hard argmax
    throws away the signal that makes confidence mean something. Brier is a proper
    scoring rule and is what pushes probabilities toward honesty rather than
    toward whichever label happens to win.
    """
    p = probs.clamp_min(1e-8)
    t = batch["target"]
    m = batch["label_mask"].float()
    n_q = batch["qmask"].sum().clamp(min=1)
    ce = -(t * p.log() * m).sum() / n_q
    brier = (((probs - t) ** 2) * m).sum() / n_q
    return ce + brier_weight * brier, ce.detach(), brier.detach()


def argmax_per_question(probs, batch):
    """-> [B, Q] index of the winning label *within* each question, -1 if absent."""
    B, Q = batch["qtype"].shape
    out = torch.full((B, Q), -1, dtype=torch.long, device=probs.device)
    conf = torch.zeros((B, Q), device=probs.device)
    for q in range(Q):
        m = (batch["group"] == q) & batch["label_mask"]
        if not m.any():
            continue
        scores = probs.masked_fill(~m, -1.0)
        best = scores.argmax(dim=-1)
        first = torch.where(m.any(-1), m.float().argmax(dim=-1), torch.zeros_like(best))
        has = m.any(-1)
        out[:, q] = torch.where(has, best - first, torch.full_like(best, -1))
        conf[:, q] = torch.where(has, scores.max(dim=-1).values,
                                 torch.zeros_like(conf[:, q]))
    return out, conf


def require_ckpt(path):
    """Load a checkpoint, or say plainly which earlier step never finished."""
    if not os.path.exists(path):
        sys.exit(
            f"'{path}' does not exist - 02_train.py has not completed. "
            "Scroll up and fix the training step; every step after it depends "
            "on the checkpoint it writes.")
    return torch.load(path, map_location="cpu")


@torch.no_grad()
def predict_distributions(model, rows, max_len, device, batch_size=8):
    """Per-question probability distributions with label names attached.

    -> [{qname: {"labels": [...], "probs": [...]}}] in the same order as `rows`.

    argmax_per_question is enough for accuracy; an ensemble needs the whole
    distribution keyed by label so it can be combined with another model that
    orders its classes differently.
    """
    from td_data import TypedDecisions, collate
    from torch.utils.data import DataLoader

    ds = TypedDecisions(rows, model.tok, max_len, model.qid, model.lid)
    dl = DataLoader(ds, batch_size=batch_size,
                    collate_fn=lambda b: collate(b, model.tok.pad_token_id))
    model.eval()
    out, at = [], 0
    for b in dl:
        n = b["input_ids"].size(0)
        b = {k: v.to(device) for k, v in b.items()}
        _, probs = model(b, apply_temperature=True)
        probs = probs.cpu()
        for i in range(n):
            enc = ds[at + i]
            row, cursor = {}, 0
            for gi, qname in enumerate(enc["qnames"]):
                labels = enc["labels"][gi]
                row[qname] = {"labels": labels,
                              "probs": probs[i, cursor:cursor + len(labels)].tolist()}
                cursor += len(labels)
            out.append(row)
        at += n
    return out
'''

FILES['metrics.py'] = r'''
"""Scoring. Accuracy is the headline; the rest is what makes confidence usable.

All functions take flat [B, M] tensors of per-label probabilities, the matching
consensus targets, and a boolean mask selecting real (non-padding) label slots.
"""
import torch


def _flat(probs, target, mask):
    return probs[mask].float(), target[mask].float()


def brier(probs, target, mask):
    """Multi-class Brier over label slots. Proper scoring rule, lower is better."""
    p, t = _flat(probs, target, mask)
    return ((p - t) ** 2).mean().item()


def nll(probs, target, mask):
    p, t = _flat(probs, target, mask)
    return (-(t * p.clamp_min(1e-8).log())).sum().item() / max(mask.sum().item(), 1)


def ece(probs, target, mask, bins=15):
    """Expected calibration error on the *chosen* label of each slot.

    Reliability is measured against the consensus probability of the label the
    model picked, which is the quantity an escalation threshold actually reads.
    """
    p, t = _flat(probs, target, mask)
    if p.numel() == 0:
        return float("nan")
    edges = torch.linspace(0, 1, bins + 1)
    total = 0.0
    for i in range(bins):
        sel = (p > edges[i]) & (p <= edges[i + 1])
        if sel.sum() == 0:
            continue
        total += (sel.float().mean() * (p[sel].mean() - t[sel].mean()).abs()).item()
    return total


def tvd(probs, target, mask, group, n_groups):
    """Total variation distance between predicted and consensus distributions.

    Accuracy only checks the argmax. TVD checks whether the whole distribution
    is right, which is what you are relying on when you route on confidence.
    """
    out = []
    for q in range(n_groups):
        m = (group == q) & mask
        if not m.any():
            continue
        d = ((probs - target).abs() * m).sum(-1) * 0.5
        out.append(d[m.any(-1)])
    return torch.cat(out).mean().item() if out else float("nan")


def risk_coverage(conf, correct, thresholds=(0.5, 0.6, 0.7, 0.8, 0.9)):
    """Accuracy among decisions the model is confident about, and how many those are.

    This is the table that decides whether you can autoroute. A model that is
    96% accurate on the 40% of cases it is sure about is useful even if its
    overall accuracy is mediocre - you escalate the rest.
    """
    rows = []
    for t in thresholds:
        sel = conf >= t
        cov = sel.float().mean().item()
        acc = correct[sel].float().mean().item() if sel.any() else float("nan")
        rows.append((t, cov, acc))
    return rows


def ece_confidence(conf, correct, bins=15):
    """Standard ECE: top-1 confidence vs. empirical correctness.

    This is the number published benchmarks quote (Jev measured at 0.144 on
    typed-decisions, 0.154 on phishing), so it is the only one that can be
    compared against them. `ece` above answers a different question - how close
    the whole predicted distribution sits to the consensus - and the two can
    disagree sharply: a model that outputs near-uniform everywhere scores well
    on distributional ECE while being useless at ranking its own errors.
    """
    conf = conf.float().flatten()
    correct = correct.float().flatten()
    if conf.numel() == 0:
        return float("nan")
    edges = torch.linspace(0, 1, bins + 1)
    total = 0.0
    for i in range(bins):
        sel = (conf > edges[i]) & (conf <= edges[i + 1])
        if sel.sum() == 0:
            continue
        total += (sel.float().mean()
                  * (conf[sel].mean() - correct[sel].mean()).abs()).item()
    return total


def reliability_bins(conf, correct, bins=10):
    """-> [(bin_centre, mean_confidence, empirical_accuracy, count)] for non-empty bins.

    The raw material of a reliability diagram: a perfectly calibrated model puts
    every point on the diagonal, above it means underconfident, below means
    overconfident.
    """
    conf = conf.float().flatten()
    correct = correct.float().flatten()
    edges = torch.linspace(0, 1, bins + 1)
    out = []
    for i in range(bins):
        sel = (conf > edges[i]) & (conf <= edges[i + 1])
        n = int(sel.sum())
        if n == 0:
            continue
        out.append((float((edges[i] + edges[i + 1]) / 2),
                    float(conf[sel].mean()), float(correct[sel].mean()), n))
    return out
'''

FILES['plots.py'] = r'''
"""Charts for the run. Reads the JSON each step writes; renders nothing it lacks.

  python plots.py              # every chart it has data for, saved to plots/
  python plots.py --show       # also display (Colab renders them inline)

Deliberately decoupled from the pipeline: the scripts dump numbers, this reads
them. You can re-style a chart without re-running a 15-minute training job.
"""
from __future__ import annotations
import argparse
import json
import pathlib

import matplotlib
import matplotlib.pyplot as plt
from matplotlib.transforms import blended_transform_factory

# --- palette -----------------------------------------------------------------
# Validated categorical slots 1-2 (worst adjacent CVD dE 24.7, normal 33.6).
# Matplotlib renders a fixed PNG that cannot follow Colab's theme, so every
# figure carries its own light surface and reads correctly on either background.
SURFACE = "#fcfcfb"
INK = "#0b0b0b"
INK_2 = "#52514e"
MUTED = "#898781"
BASELINE = "#c3c2b7"
S1 = "#2a78d6"    # blue
S2 = "#eb6834"    # orange
GOOD = "#0ca30c"
CRITICAL = "#d03b3b"

JEV_ACC = 0.727
JEV_ECE = 0.144


def style():
    matplotlib.rcParams.update({
        "figure.facecolor": SURFACE, "axes.facecolor": SURFACE,
        "savefig.facecolor": SURFACE,
        "text.color": INK, "axes.labelcolor": INK_2,
        "xtick.color": MUTED, "ytick.color": MUTED,
        "axes.edgecolor": BASELINE, "axes.linewidth": 1.0,
        "axes.spines.top": False, "axes.spines.right": False,
        "axes.grid": True, "grid.color": BASELINE, "grid.alpha": 0.45,
        "grid.linewidth": 0.8,
        "lines.linewidth": 2.0, "lines.markersize": 8,
        "font.size": 11, "axes.titlesize": 13, "axes.titleweight": "bold",
        "axes.titlelocation": "left", "axes.titlepad": 12,
        "figure.dpi": 120,
    })


def hline_label(ax, y, text, color=None):
    """Caption a horizontal reference line at the right edge, just above it.

    Anchoring in data coordinates put the label outside the axes on some
    panels and it silently vanished; x here is axes-relative, y is data.
    """
    tr = blended_transform_factory(ax.transAxes, ax.transData)
    ax.annotate(text, xy=(0.995, y), xycoords=tr, xytext=(0, 4),
                textcoords="offset points", ha="right", va="bottom",
                fontsize=9, color=color or CRITICAL, fontweight="bold")


def load(name):
    p = pathlib.Path(name)
    return json.loads(p.read_text(encoding="utf-8")) if p.exists() else None


def _finish(fig, ax_or_axes, name, subtitle, outdir, show):
    axes = ax_or_axes if isinstance(ax_or_axes, (list, tuple)) else [ax_or_axes]
    for ax in axes:
        ax.set_axisbelow(True)
    if subtitle:
        fig.text(0.0, 1.0, subtitle, ha="left", va="bottom",
                 fontsize=10, color=INK_2, transform=fig.transFigure)
    fig.tight_layout()
    out = pathlib.Path(outdir) / name
    out.parent.mkdir(exist_ok=True)
    fig.savefig(out, bbox_inches="tight")
    print(f"  {out}")
    if show:
        plt.show()
    else:
        plt.close(fig)


# --- 1. scoreboard ------------------------------------------------------------
def scoreboard(base, res, outdir, show, ens=None):
    """Magnitude across methods -> horizontal bars, one series, no legend.

    Jev is a reference line rather than a bar: it is the thing being compared
    against, not another entry in the same list.
    """
    rows = []
    if base:
        rows += [("Majority class", base["majority"], False),
                 ("Single annotator", base["annotator"], False)]
        if base.get("frozen") is not None:
            rows.append(("Frozen encoder + logreg", base["frozen"], False))
    if res:
        rows.append(("Fine-tuned model", res["accuracy"], False))
    if ens:
        # the headline configuration, so it is the one painted as ours
        best = max(ens["test"].items(), key=lambda kv: kv[1]["accuracy"])
        rows.append(("Ensemble (this repo)", best[1]["accuracy"], True))
    elif rows:
        rows[-1] = (rows[-1][0], rows[-1][1], True)
    if not rows:
        return
    rows.sort(key=lambda r: r[1])

    fig, ax = plt.subplots(figsize=(8.0, 0.62 * len(rows) + 2.0))
    ys = range(len(rows))
    colors = [S1 if mine else BASELINE for _, _, mine in rows]
    ax.barh(list(ys), [v for _, v, _ in rows], height=0.62, color=colors)

    # Values sit in their own column past every bar and past the Jev rule, so a
    # number can never land on top of the line it is being compared against.
    top = max(max(v for _, v, _ in rows), JEV_ACC)
    label_x = top * 1.06
    for y, (_, v, mine) in zip(ys, rows):
        ax.text(label_x, y, f"{v:.3f}", va="center", fontsize=11,
                color=INK if mine else INK_2,
                fontweight="bold" if mine else "normal")

    ax.axvline(JEV_ACC, color=CRITICAL, linewidth=2, linestyle=(0, (5, 3)), zorder=3)
    # Caption the rule above the bars: below the axis it lands in the tick
    # labels, and level with a bar it lands on that bar's value.
    head = len(rows) - 0.5 + 0.80
    ax.annotate(f"TypeSafe Jev {JEV_ACC:.3f}  ", xy=(JEV_ACC, len(rows) - 0.5 + 0.12),
                xytext=(-4, 0), textcoords="offset points",
                ha="right", va="bottom", color=CRITICAL, fontsize=10,
                fontweight="bold")

    ax.set_yticks(list(ys), [n for n, _, _ in rows], color=INK_2, fontsize=11)
    ax.set_ylim(-0.65, head)
    ax.set_xlim(0, top * 1.20)
    n = (res or {}).get("n_cases")
    ax.set_xlabel(f"accuracy on the {n}-case test split" if n
                  else "accuracy on the test split")
    ax.set_title("Accuracy vs TypeSafe Jev")
    ax.grid(axis="y", visible=False)
    _finish(fig, ax, "1_scoreboard.png",
            "Blue is the shipped configuration. Grey needs little or no training.",
            outdir, show)


# --- 2. training curve --------------------------------------------------------
def training(hist, outdir, show):
    """Two measures on different scales -> two panels, never two y-axes."""
    if not hist:
        return
    ep = [h["epoch"] for h in hist]
    fig, (a1, a2) = plt.subplots(1, 2, figsize=(10.5, 3.9))

    a1.plot(ep, [h["loss"] for h in hist], color=S1, marker="o",
            markeredgecolor=SURFACE, markeredgewidth=2)
    a1.set_title("Training loss")
    a1.set_xlabel("epoch")

    acc = [h["val_acc"] for h in hist]
    a2.plot(ep, acc, color=S2, marker="o",
            markeredgecolor=SURFACE, markeredgewidth=2)
    best = max(range(len(acc)), key=lambda i: acc[i])
    # Anchor the callout away from whichever edge the best epoch sits against.
    at_end = best >= len(acc) - 1
    a2.annotate(f"best {acc[best]:.3f}", (ep[best], acc[best]),
                textcoords="offset points",
                xytext=(-10 if at_end else 0, 12),
                ha="right" if at_end else "center",
                fontsize=10, color=INK, fontweight="bold")
    span = (max(acc) - min(acc)) or 0.02
    a2.set_ylim(min(acc) - span * 0.18, max(acc) + span * 0.38)
    a2.set_title("Validation accuracy")
    a2.set_xlabel("epoch")

    for ax in (a1, a2):
        ax.set_xticks(ep)
        ax.grid(axis="x", visible=False)
    _finish(fig, [a1, a2], "2_training.png",
            "Loss still falling at the last epoch means it is undertrained.",
            outdir, show)


# --- 3. reliability -----------------------------------------------------------
def reliability(res, outdir, show):
    """Calibration: two series (before/after) -> legend plus direct labels."""
    if not res or not res.get("reliability_raw"):
        return
    fig, ax = plt.subplots(figsize=(6.4, 5.4))
    ax.plot([0, 1], [0, 1], color=BASELINE, linewidth=1.5,
            linestyle=(0, (4, 3)), zorder=1)
    ax.text(0.97, 0.93, "perfectly calibrated", color=MUTED, fontsize=9,
            ha="right", rotation=38, rotation_mode="anchor")

    for key, colour, label in (("reliability_raw", BASELINE, "before"),
                               ("reliability_cal", S1, "after temperature")):
        pts = res.get(key)
        if not pts:
            continue
        ax.plot([p[1] for p in pts], [p[2] for p in pts], color=colour,
                marker="o", markeredgecolor=SURFACE, markeredgewidth=2,
                label=label, zorder=3 if colour == S1 else 2)

    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.set_xlabel("confidence the model reported")
    ax.set_ylabel("how often it was actually right")
    ax.set_title("Reliability")
    ax.legend(frameon=False, loc="lower right", labelcolor=INK_2)
    _finish(fig, ax, "3_reliability.png",
            "Below the diagonal = overconfident. Above = underconfident.",
            outdir, show)


# --- 4. coverage --------------------------------------------------------------
def coverage(res, outdir, show):
    """One series -> no legend; label the points that carry the decision."""
    rows = (res or {}).get("coverage")
    if not rows:
        return
    rows = [r for r in rows if r[1] > 0]
    if not rows:
        return
    fig, ax = plt.subplots(figsize=(7.2, 4.6))
    cov = [r[1] for r in rows]
    acc = [r[2] for r in rows]
    ax.plot(cov, acc, color=S1, marker="o",
            markeredgecolor=SURFACE, markeredgewidth=2, zorder=3)
    # High thresholds crowd together at low coverage, so alternate the callout
    # above and below whenever two points sit close on the x axis.
    span = max(max(cov) - min(cov), 1e-6)
    last_x, above = None, True
    for t, c, a in sorted(rows, key=lambda r: r[1]):
        if last_x is not None and (c - last_x) / span < 0.12:
            above = not above
        else:
            above = True
        ax.annotate(f"≥{t:.2f}", (c, a), textcoords="offset points",
                    xytext=(0, 11 if above else -18), ha="center",
                    fontsize=9, color=INK_2)
        last_x = c

    overall = res.get("accuracy")
    if overall:
        ax.axhline(overall, color=BASELINE, linewidth=1.5, linestyle=(0, (4, 3)))
        hline_label(ax, overall, f"all decisions {overall:.3f}", MUTED)

    ax.set_xlabel("coverage - share of decisions kept")
    ax.set_ylabel("accuracy on the decisions kept")
    ax.set_title("Can you route on the confidence?")
    ax.set_xlim(0, 1.02)
    _finish(fig, ax, "4_coverage.png",
            "Up and to the left is the useful shape: high accuracy on a band "
            "you can autoroute, the rest escalated.", outdir, show)


# --- 5. per type --------------------------------------------------------------
def per_type(res, outdir, show):
    """Accuracy and ECE are different scales and opposite polarity -> two panels."""
    rows = (res or {}).get("per_type")
    if not rows:
        return
    names = [r["type"] for r in rows]
    fig, (a1, a2) = plt.subplots(1, 2, figsize=(10.5, 4.0))
    xs = range(len(names))

    a1.bar(list(xs), [r["accuracy"] for r in rows], width=0.58, color=S1)
    for x, r in zip(xs, rows):
        a1.text(x, r["accuracy"] + 0.012, f"{r['accuracy']:.3f}", ha="center",
                fontsize=10, color=INK_2)
    a1.set_title("Accuracy by question type")
    a1.set_ylim(0, 1.08)

    a2.bar(list(xs), [r["ece"] for r in rows], width=0.58, color=S2)
    for x, r in zip(xs, rows):
        a2.text(x, r["ece"] + 0.004, f"{r['ece']:.3f}", ha="center",
                fontsize=10, color=INK_2)
    a2.axhline(JEV_ECE, color=CRITICAL, linewidth=2, linestyle=(0, (5, 3)))
    hline_label(a2, JEV_ECE, f"Jev {JEV_ECE:.3f}")
    a2.set_ylim(0, max(JEV_ECE, max(r["ece"] for r in rows)) * 1.30)
    a2.set_title("Calibration error by type (lower is better)")

    for ax in (a1, a2):
        ax.set_xticks(list(xs), names, color=INK_2)
        ax.grid(axis="x", visible=False)
    _finish(fig, [a1, a2], "5_per_type.png",
            "Jev's own error runs opposite ways by type - that is why each gets "
            "its own temperature.", outdir, show)


# --- 6. ensemble ---------------------------------------------------------------
def ensemble(ens, outdir, show):
    """Two panels: the weight sweep that chose w, and what it bought on test."""
    if not ens:
        return
    fig, (a1, a2) = plt.subplots(1, 2, figsize=(11.0, 4.2))

    ws = [c["w"] for c in ens["val_curve"]]
    accs = [c["accuracy"] for c in ens["val_curve"]]
    a1.plot(ws, accs, color=S1)
    bw = ens["best_w"]
    a1.scatter([bw], [max(accs)], color=S1, zorder=4, s=70,
               edgecolor=SURFACE, linewidth=2)
    a1.annotate(f"w={bw:.2f}", (bw, max(accs)), textcoords="offset points",
                xytext=(0, 12), ha="center", fontsize=10, color=INK,
                fontweight="bold")
    a1.set_xlabel("w   (1.0 = neural model only, 0.0 = frozen probe only)")
    a1.set_ylabel("validation accuracy")
    a1.set_title("Choosing the blend weight")

    names = list(ens["test"])
    vals = [ens["test"][n]["accuracy"] for n in names]
    eces = [ens["test"][n]["ece"] for n in names]
    # Ties on accuracy break toward the better-calibrated configuration, which
    # is the one actually shipped - sharpening moves ECE, never the argmax.
    best = max(range(len(vals)), key=lambda i: (vals[i], -eces[i]))
    colors = [S1 if i == best else BASELINE for i in range(len(names))]
    a2.bar(range(len(names)), vals, width=0.58, color=colors)
    # Labels sit inside the bars: above them they collide with the Jev rule,
    # which on this chart runs only a hair above the tallest bar.
    for i, v in enumerate(vals):
        a2.text(i, v - 0.028, f"{v:.3f}", ha="center", va="top", fontsize=10,
                color=SURFACE if i == best else INK_2,
                fontweight="bold" if i == best else "normal")
        a2.text(i, v - 0.075, f"ECE {eces[i]:.3f}", ha="center", va="top",
                fontsize=8, color=SURFACE if i == best else MUTED)
    a2.axhline(JEV_ACC, color=CRITICAL, linewidth=2, linestyle=(0, (5, 3)))
    hline_label(a2, JEV_ACC, f"Jev {JEV_ACC:.3f}")
    a2.set_xticks(range(len(names)),
                  [n.strip().replace(" ", chr(10)) for n in names],
                  color=INK_2, fontsize=9)
    a2.set_ylim(0, max(max(vals), JEV_ACC) * 1.16)
    a2.set_title("Test accuracy")
    a2.grid(axis="x", visible=False)

    _finish(fig, [a1, a2], "6_ensemble.png",
            "Combining two models that fail differently, weight chosen on "
            "validation and never on test.", outdir, show)


def main(outdir="plots", show=False):
    style()
    base, hist, res = load("baselines.json"), load("history.json"), load("results.json")
    ens = load("ensemble.json")
    if not any((base, hist, res, ens)):
        raise SystemExit(
            "no run artifacts found - run 01_ceiling.py / 02_train.py / "
            "04_eval.py first; each writes the JSON this reads.")
    # Artifacts from different runs plot happily side by side and the chart
    # gives no hint that the bars are not comparable.
    cfgs = {k: v.get("config") for k, v in (("baselines", base), ("results", res))
            if v and v.get("config")}
    if len(set(cfgs.values())) > 1:
        print(f"  ! mixed runs: {cfgs} - the scoreboard compares bars measured on "
              "different data. Rerun 01_ceiling.py and 04_eval.py with the same "
              "--config before quoting it.")
    if res and res.get("n_cases") and res["n_cases"] < 400 and res.get("config") == "all":
        print(f"  ! results.json covers only {res['n_cases']} of the 400 test "
              "cases (--limit was set) - not comparable to Jev's published number.")

    print("charts written:")
    scoreboard(base, res, outdir, show, ens)
    training(hist, outdir, show)
    reliability(res, outdir, show)
    coverage(res, outdir, show)
    per_type(res, outdir, show)
    ensemble(ens, outdir, show)


if __name__ == "__main__":
    p = argparse.ArgumentParser()
    p.add_argument("--outdir", default="plots")
    p.add_argument("--show", action="store_true")
    a = p.parse_args()
    main(a.outdir, a.show)
'''

FILES['smoke_test.py'] = r'''
"""Cheap correctness checks that do not need a GPU or a trained checkpoint.

  python smoke_test.py            # tokenization + batching, tokenizer download only
  python smoke_test.py --all      # validate EVERY row of train+test (no model)
  python smoke_test.py --full     # also runs an untrained forward pass (~600MB)

What it verifies: that every marker position really lands on a <<l>> token, that
grouped softmax sums to 1 inside each question and stays 0 on padding, that gold
targets line up with the labels they are supposed to score, and that the metrics
agree with hand-computed values.
"""
import argparse
import torch
from transformers import AutoTokenizer
from td_data import (load_split, TypedDecisions, collate, add_markers, encode,
                     pad_cat, L_TOKEN)
from typed_schema import iter_labels
from model import grouped_softmax, DEFAULT_ENCODER
from metrics import ece, brier

MAX_LEN = 1024


def validate_all(tok, qid, lid):
    """Encode every row in the dataset and check the schema invariants.

    Worth its runtime: 10% of typed-decisions is bare `noul` questions carrying
    no `criteria` key at all, and that shape does not appear in the first rows
    of the first workflow - so a 6-row smoke test passes while a full training
    run dies on row 300. Any assumption about question shape gets checked here,
    against all of it, before a GPU is involved.
    """
    from typed_schema import TYPES
    n_q = n_bad = 0
    for split in ("train", "test"):
        rows = load_split(split)
        for r in rows:
            enc = encode(r, tok, MAX_LEN, qid, lid)
            b = collate([enc], tok.pad_token_id)
            got = b["input_ids"].gather(1, b["label_pos"])[b["label_mask"]]
            assert (got == lid).all(), f"{r['id']}: marker misalignment"
            for gi, qname in enumerate(enc["qnames"]):
                q = r["questions"][qname]
                assert q["type"] in TYPES, f"{r['id']}/{qname}: type {q['type']}"
                labels = set(enc["labels"][gi])
                gold_keys = set(r["gold"][qname]["probabilities"])
                if labels != gold_keys:
                    print(f"  ! {r['id']}/{qname}: labels {sorted(labels)} "
                          f"!= gold keys {sorted(gold_keys)}")
                    n_bad += 1
                n_q += 1
    print(f"ok  encoded every row; {n_q} questions checked, {n_bad} label/gold "
          f"mismatches")
    assert n_bad == 0, "gold distributions do not key on the labels we score"


def main(full, check_all=False):
    rows = load_split("test", limit=6)
    print(f"loaded {len(rows)} rows  ({rows[0]['id']})")

    tok = AutoTokenizer.from_pretrained(DEFAULT_ENCODER)
    _, qid, lid = add_markers(tok)
    ds = TypedDecisions(rows, tok, MAX_LEN, qid, lid)
    batch = collate([ds[i] for i in range(len(ds))], tok.pad_token_id)

    # 1. every recorded label position must be a marker token
    ids, pos, mask = batch["input_ids"], batch["label_pos"], batch["label_mask"]
    at_marker = ids.gather(1, pos)[mask]
    assert (at_marker == lid).all(), "label_pos does not point at <<l>> markers"
    print(f"ok  {mask.sum().item()} label positions all land on {L_TOKEN}")

    # 2. label count must match the schema
    expected = sum(len(iter_labels(r["questions"][q]))
                   for r in rows for q in sorted(r["questions"]))
    assert mask.sum().item() == expected, f"{mask.sum().item()} != {expected}"
    print(f"ok  label count matches schema ({expected})")

    # 3. gold targets sum to 1 within each question, 0 on padding
    Q = batch["qtype"].size(1)
    for q in range(Q):
        m = (batch["group"] == q) & mask
        if not m.any():
            continue
        s = (batch["target"] * m).sum(-1)[m.any(-1)]
        assert torch.allclose(s, torch.ones_like(s), atol=2e-2), f"q{q} targets sum {s}"
    assert (batch["target"][~mask] == 0).all()
    print("ok  consensus targets normalised per question, zero on padding")

    # 4. grouped softmax is a valid distribution per question
    logits = torch.randn_like(batch["target"])
    probs = grouped_softmax(logits, batch["group"], mask, Q)
    assert (probs[~mask] == 0).all(), "softmax leaked into padding"
    for q in range(Q):
        m = (batch["group"] == q) & mask
        if not m.any():
            continue
        s = (probs * m).sum(-1)[m.any(-1)]
        assert torch.allclose(s, torch.ones_like(s), atol=1e-5), f"q{q} probs sum {s}"
    print("ok  grouped softmax normalises within questions only")

    # 5. metrics sanity - a perfect predictor scores 0 on both
    assert abs(brier(batch["target"], batch["target"], mask)) < 1e-6
    assert ece(batch["target"], batch["target"], mask) < 1e-6
    print("ok  Brier and ECE are 0 for a perfect predictor")

    seq = batch["input_ids"].size(1)
    print(f"\nbatch: {ids.size(0)} cases, {seq} tokens, {mask.sum().item()} labels")

    if full:
        from model import JevLite, decision_loss, argmax_per_question
        m = JevLite(DEFAULT_ENCODER)
        m.eval()
        with torch.no_grad():
            _, probs = m(batch)
        loss, ce, br = decision_loss(probs, batch)
        pred, conf = argmax_per_question(probs, batch)
        gold, _ = argmax_per_question(batch["target"], batch)
        print(f"ok  untrained forward pass: loss {loss:.4f} (ce {ce:.4f} brier {br:.4f})")
        print(f"    pred {pred[0].tolist()}  gold {gold[0].tolist()}  "
              f"conf {[round(c,3) for c in conf[0].tolist()]}")
        assert probs.shape == batch["target"].shape
        print("ok  output shape matches target shape")

    # 6. batches drawn from different workflows disagree on label width
    mixed, seen = [], set()
    for r in load_split("test"):
        if r["workflow"] not in seen:
            seen.add(r["workflow"])
            mixed.append(r)
        if len(seen) >= 4:
            break
    widths, parts = set(), []
    for r in mixed:
        bb = collate([encode(r, tok, MAX_LEN, qid, lid)], tok.pad_token_id)
        widths.add(bb["target"].size(1))
        parts.append(bb)
    assert len(widths) > 1, ("workflows now agree on label count - this test no "
                             "longer guards anything, rewrite it")
    stacked = pad_cat([b["target"] for b in parts])
    smask = pad_cat([b["label_mask"] for b in parts], False)
    assert stacked.shape == smask.shape == (len(parts), max(widths))
    assert (stacked[~smask] == 0).all()
    print(f"ok  pad_cat joins ragged batches (widths {sorted(widths)}); "
          "a plain torch.cat raises here")

    if check_all:
        print("\nvalidating every row of train + test...")
        validate_all(tok, qid, lid)

    print("\nall checks passed")


if __name__ == "__main__":
    p = argparse.ArgumentParser()
    p.add_argument("--full", action="store_true")
    p.add_argument("--all", action="store_true")
    a = p.parse_args()
    main(a.full, a.all)
'''

FILES['01_ceiling.py'] = r'''
"""Step 1 - reference points and baselines, before you train anything. Costs $0.

  annotator   Mean probability that a randomly drawn annotator agrees with the
              consensus label. This is what ONE human rater scores against the
              consensus - a reference point, NOT a ceiling. A model that always
              predicts the consensus argmax scores 1.000, so exceeding this
              number is normal and expected; Jev's 0.727 already does.
  ambiguous   Share of decisions where the top two labels are within 0.1. On
              these the consensus label is close to a coin flip, so accuracy
              there is mostly luck. The `clear` column is the honest signal.
  majority    Always answer the most common label for that (workflow, question).
  frozen      Frozen encoder embeddings + logistic regression. This is the
              baseline that beat Jev by 10 points on Banking77 at 44x the speed.
              If your fine-tune cannot clear it, ship this instead.
"""
import argparse
import json
import numpy as np
from collections import defaultdict
from td_data import load_split
# state_to_text now lives behind probe.py; nothing here needs it directly

AMBIGUOUS_MARGIN = 0.1     # same threshold the dataset's own tooling uses


def annotator_agreement(rows):
    """Expected accuracy of a single random annotator judged against consensus."""
    per_type, allv = defaultdict(list), []
    for r in rows:
        for qn, g in r["gold"].items():
            top = max(g["probabilities"].values())
            per_type[g["type"]].append(top)
            allv.append(top)
    return float(np.mean(allv)), {k: float(np.mean(v)) for k, v in per_type.items()}


def ambiguous_fraction(rows):
    """Share of decisions whose consensus label is nearly a tie."""
    amb = tot = 0
    for r in rows:
        for qn, g in r["gold"].items():
            p = sorted(g["probabilities"].values(), reverse=True)
            margin = p[0] - (p[1] if len(p) > 1 else 0.0)
            amb += int(margin < AMBIGUOUS_MARGIN)
            tot += 1
    return amb / tot, tot


def majority(train, test):
    tally = defaultdict(lambda: defaultdict(int))
    for r in train:
        for qn, g in r["gold"].items():
            tally[(r["workflow"], qn)][g["label"]] += 1
    best = {k: max(v, key=v.get) for k, v in tally.items()}
    hit = tot = 0
    for r in test:
        for qn, g in r["gold"].items():
            tot += 1
            hit += int(best.get((r["workflow"], qn)) == g["label"])
    return hit / tot


def frozen_probe(train, test, encoder, max_len, batch_size=16):
    """Accuracy of embeddings + one logistic regression per (workflow, question).

    Shares its implementation with 07_ensemble.py via probe.py - two copies of
    the baseline would eventually disagree, and then the comparison is fiction.
    """
    from probe import embed_states, fit_probes

    Xtr = embed_states(train, encoder, max_len, batch_size)
    Xte = embed_states(test, encoder, max_len, batch_size)
    probes = fit_probes(train, Xtr)
    hit = tot = 0
    for i, r in enumerate(test):
        for qn, g in r["gold"].items():
            clf = probes.get((r["workflow"], qn))
            if clf is None:
                continue
            pred = clf if isinstance(clf, str) else clf.predict(Xte[i:i + 1])[0]
            hit += int(pred == g["label"])
            tot += 1
    return hit / max(tot, 1)


if __name__ == "__main__":
    p = argparse.ArgumentParser()
    p.add_argument("--encoder", default="answerdotai/ModernBERT-base")
    p.add_argument("--max-len", type=int, default=512)
    p.add_argument("--skip-frozen", action="store_true")
    p.add_argument("--config", default="all",
                   help="one workflow (customer_service, invoice_processing, "
                        "security_incidents, agent_trace_observability) instead "
                        "of all four - a domain specialist is where a small model wins")
    a = p.parse_args()

    train, test = load_split("train", a.config), load_split("test", a.config)
    n_dec = sum(len(r["gold"]) for r in test)
    print(f"train {len(train)} cases | test {len(test)} cases / {n_dec} decisions\n")

    c, per_type = annotator_agreement(test)
    spread = "  ".join(f"{k}={v:.3f}" for k, v in sorted(per_type.items()))
    amb, tot = ambiguous_fraction(test)
    print(f"  single annotator         {c:.3f}   {spread}")
    print(f"  ambiguous decisions      {amb:.1%}   "
          f"({int(amb*tot)}/{tot} within {AMBIGUOUS_MARGIN} of a tie)")
    maj = majority(train, test)
    print(f"  majority baseline        {maj:.3f}")
    fz = None
    if not a.skip_frozen:
        fz = frozen_probe(train, test, a.encoder, a.max_len)
        print(f"  frozen encoder + logreg  {fz:.3f}   (no fine-tuning)")

    json.dump({"annotator": c, "annotator_per_type": per_type, "ambiguous": amb,
               "majority": maj, "frozen": fz, "n_decisions": tot,
               "config": a.config},
              open("baselines.json", "w"), indent=1)

    print(f"\n  TypeSafe Jev 1.13.0      0.727   (published)")
    print("\nTargets: clear the frozen probe to justify fine-tuning at all, then")
    print("clear 0.727 to beat Jev. 'single annotator' is what one human rater")
    print("scores, not a cap - models routinely exceed it and Jev already does.")
    print(f"But {amb:.0%} of decisions are near-ties, so the last few points of")
    print("headline accuracy are largely luck. Judge on the calibration and")
    print("coverage tables in 04_eval.py, not on accuracy alone.")
'''

FILES['02_train.py'] = r'''
"""Step 2 - fine-tune JevLite on typed-decisions. Colab T4, ~15 min.

  python 02_train.py --epochs 6

Trains against the consensus *distribution* (soft CE + Brier), not the argmax.
That is the whole calibration story: a model taught that a 55/45 case is 55/45
reports 0.55, while a model taught it is "true" reports 0.95 and is then wrong
45% of the time while claiming near-certainty.
"""
import argparse
import json
import time
import torch
from torch.utils.data import DataLoader
from transformers import (get_cosine_schedule_with_warmup,
                          get_linear_schedule_with_warmup)
from td_data import load_split, TypedDecisions, collate, val_split
from model import JevLite, decision_loss, argmax_per_question, DEFAULT_ENCODER



def accuracy(model, dl, dev, temp=False):
    model.eval()
    hit = tot = 0
    with torch.no_grad():
        for b in dl:
            b = {k: v.to(dev) for k, v in b.items()}
            _, probs = model(b, apply_temperature=temp)
            pred, _ = argmax_per_question(probs, b)
            gold, _ = argmax_per_question(b["target"], b)
            m = b["qmask"]
            hit += ((pred == gold) & m).sum().item()
            tot += m.sum().item()
    return hit / max(tot, 1)


if __name__ == "__main__":
    p = argparse.ArgumentParser()
    p.add_argument("--encoder", default=DEFAULT_ENCODER)
    p.add_argument("--epochs", type=int, default=20,
                   help="6 was measurably too few - validation accuracy was still "
                        "climbing steeply when the first run stopped")
    p.add_argument("--patience", type=int, default=6,
                   help="stop after this many epochs with no validation gain; "
                        "0 disables. The best checkpoint is kept either way")
    p.add_argument("--schedule", default="cosine", choices=["cosine", "linear"])
    p.add_argument("--bs", type=int, default=4)
    p.add_argument("--accum", type=int, default=4)
    p.add_argument("--max-len", type=int, default=1024)
    p.add_argument("--lr", type=float, default=3e-5)
    p.add_argument("--head-lr", type=float, default=1e-3)
    p.add_argument("--brier", type=float, default=1.0)
    p.add_argument("--out", default="jevlite.pt")
    p.add_argument("--config", default="all",
                   help="one workflow (customer_service, invoice_processing, "
                        "security_incidents, agent_trace_observability) instead "
                        "of all four - a domain specialist is where a small model wins")
    p.add_argument("--limit", type=int, default=None,
                   help="tiny run to verify the loop before spending GPU time")
    a = p.parse_args()

    dev = "cuda" if torch.cuda.is_available() else "cpu"
    model = JevLite(a.encoder).to(dev)
    tok = model.tok
    tr_rows, va_rows = val_split(load_split("train", a.config, limit=a.limit))

    def mk(rows):
        return TypedDecisions(rows, tok, a.max_len, model.qid, model.lid)

    def coll(b):
        return collate(b, tok.pad_token_id)

    tl = DataLoader(mk(tr_rows), batch_size=a.bs, shuffle=True, collate_fn=coll)
    vl = DataLoader(mk(va_rows), batch_size=a.bs, collate_fn=coll)
    print(f"device {dev} | train {len(tr_rows)} | val {len(va_rows)} "
          f"| effective batch {a.bs * a.accum}")

    head = [q for n, q in model.named_parameters() if n.startswith("score.")]
    enc = [q for n, q in model.named_parameters()
           if not n.startswith("score.") and q.requires_grad]
    opt = torch.optim.AdamW([{"params": enc, "lr": a.lr},
                             {"params": head, "lr": a.head_lr}], weight_decay=0.01)
    steps = max(1, (len(tl) // a.accum) * a.epochs)
    mk_sched = (get_cosine_schedule_with_warmup if a.schedule == "cosine"
                else get_linear_schedule_with_warmup)
    sched = mk_sched(opt, int(0.1 * steps), steps)
    scaler = torch.amp.GradScaler("cuda", enabled=(dev == "cuda"))

    best, history, stale = -1.0, [], 0
    # An epoch is minutes long and silence is indistinguishable from a hang, so
    # emit a heartbeat. Colab in particular gives no other signal that a cell is
    # alive rather than wedged.
    every = max(1, len(tl) // 10)
    for ep in range(a.epochs):
        model.train()
        run = 0.0
        t_ep = time.time()
        for i, b in enumerate(tl):
            b = {k: v.to(dev) for k, v in b.items()}
            with torch.autocast("cuda", dtype=torch.float16, enabled=(dev == "cuda")):
                _, probs = model(b)
            loss, ce, br = decision_loss(probs, b, a.brier)
            scaler.scale(loss / a.accum).backward()
            run += loss.item()
            if (i + 1) % every == 0:
                done = (i + 1) / len(tl)
                el = time.time() - t_ep
                print(f"  epoch {ep+1}  {done:4.0%}  loss {run/(i+1):.4f}  "
                      f"{el:.0f}s elapsed, ~{el/done - el:.0f}s left", flush=True)
            if (i + 1) % a.accum == 0:
                scaler.unscale_(opt)
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                scaler.step(opt)
                scaler.update()
                sched.step()
                opt.zero_grad(set_to_none=True)
        acc = accuracy(model, vl, dev)
        star = "  *" if acc > best else ""
        print(f"epoch {ep+1}/{a.epochs}  loss {run/len(tl):.4f}  val acc {acc:.4f}{star}")
        # Written every epoch, not at the end: a reclaimed Colab session still
        # leaves you the curve that tells you whether to train longer.
        history.append({"epoch": ep + 1, "loss": run / len(tl), "val_acc": acc})
        json.dump(history, open("history.json", "w"), indent=1)

        if acc > best:
            stale, best = 0, acc
            torch.save({"state_dict": model.state_dict(), "encoder": a.encoder,
                        "max_len": a.max_len, "val_acc": acc}, a.out)
        else:
            stale += 1
            if a.patience and stale >= a.patience:
                print(f"\nno gain for {stale} epochs - stopping at epoch {ep+1}. "
                      f"The best checkpoint ({best:.4f}) is what was kept.")
                break
    print(f"\nbest val accuracy {best:.4f} -> {a.out}")
    print("Next: python 03_calibrate.py   (fits per-type temperature on val)")
'''

FILES['03_calibrate.py'] = r'''
"""Step 3 - fit one temperature per primitive type on the validation split.

This is the cheapest win available and it targets Jev's weakest measured claim.
Independent testing found Jev's miscalibration runs in *opposite directions* by
type: choice and score overconfident (refit T around 3.3), noul underconfident
(T around 0.66). A single global temperature cannot repair both at once, so we
fit three and store them in the checkpoint.

Runs on CPU in seconds. Nothing here touches the encoder weights.
"""
import argparse
import torch
from torch.utils.data import DataLoader
from td_data import load_split, TypedDecisions, collate, val_split, pad_cat
from model import JevLite, grouped_softmax, argmax_per_question, require_ckpt
from typed_schema import TYPES
from metrics import ece, ece_confidence, brier, nll


def collect(model, dl, dev):
    """Cache raw logits once so the temperature search is a pure CPU fit."""
    out = []
    model.eval()
    with torch.no_grad():
        for b in dl:
            b = {k: v.to(dev) for k, v in b.items()}
            lg = model.logits(b["input_ids"], b["attention_mask"], b["label_pos"])
            out.append({k: v.cpu() for k, v in b.items()} | {"logits": lg.float().cpu()})
    return out


def apply_temp(batch, log_temp):
    g = batch["group"].clamp(min=0)
    t = batch["qtype"].clamp(min=0).gather(1, g)
    lg = batch["logits"] / log_temp.exp()[t]
    return grouped_softmax(lg, batch["group"], batch["label_mask"],
                           batch["qtype"].size(1))


T_MIN, T_MAX = 0.25, 5.0
MIN_DECISIONS = 30


def _bounded(raw):
    """T in [T_MIN, T_MAX] via sigmoid, so the search cannot run to infinity.

    Unbounded NLL minimisation against *soft* targets has a degenerate optimum:
    when the consensus is near-uniform, T -> inf flattens every prediction to
    uniform and scores well on NLL while destroying the confidence signal the
    whole point of this step was to make trustworthy. Ask for T=400 and you get
    a model that is beautifully calibrated and useless for routing.
    """
    return T_MIN + (T_MAX - T_MIN) * torch.sigmoid(raw)


def type_counts(cached, n_types=len(TYPES)):
    counts = torch.zeros(n_types, dtype=torch.long)
    for b in cached:
        ty = b["qtype"][b["qmask"]]
        for i in range(n_types):
            counts[i] += int((ty == i).sum())
    return counts


def fit(cached, n_types=len(TYPES), iters=200):
    # sigmoid(0) = 0.5 -> T starts at the midpoint of the bounded range; offset
    # so the initial value is 1.0 (a no-op temperature).
    init = torch.log(torch.tensor((1.0 - T_MIN) / (T_MAX - 1.0)))
    raw = torch.full((n_types,), float(init), requires_grad=True)
    opt = torch.optim.LBFGS([raw], lr=0.1, max_iter=iters)

    def closure():
        opt.zero_grad()
        loss = 0.0
        for b in cached:
            p = apply_temp(b, _bounded(raw).log()).clamp_min(1e-8)
            m = b["label_mask"].float()
            loss = loss - (b["target"] * p.log() * m).sum() / b["qmask"].sum()
        loss.backward()
        return loss

    opt.step(closure)
    temps = _bounded(raw).detach()

    # A type with too few validation decisions gets T=1 rather than a fit to noise.
    counts = type_counts(cached, n_types)
    for i, c in enumerate(counts):
        if c < MIN_DECISIONS:
            print(f"  ! {TYPES[i]}: only {int(c)} validation decisions "
                  f"(< {MIN_DECISIONS}) - leaving T=1.0 instead of fitting noise")
            temps[i] = 1.0
    at_bound = [(TYPES[i], float(t)) for i, t in enumerate(temps)
                if t <= T_MIN * 1.01 or t >= T_MAX * 0.99]
    for name, t in at_bound:
        print(f"  ! {name}: T={t:.2f} hit the bound - the soft targets for this "
              "type are close to uniform; treat its confidence with suspicion")
    return temps.log()


def report(cached, log_temp, tag):
    ps, ts, ms, tys, confs, corrects, qts = [], [], [], [], [], [], []
    for b in cached:
        p = apply_temp(b, log_temp)
        pred, conf = argmax_per_question(p, b)
        gold, _ = argmax_per_question(b["target"], b)
        m = b["qmask"]
        confs.append(conf[m]); corrects.append((pred == gold)[m])
        qts.append(b["qtype"][m])
        ps.append(p); ts.append(b["target"]); ms.append(b["label_mask"])
        tys.append(b["qtype"].clamp(min=0).gather(1, b["group"].clamp(min=0)))
    P, T = pad_cat(ps), pad_cat(ts)
    M, TY = pad_cat(ms, False), pad_cat(tys)
    C, OK, QT = torch.cat(confs), torch.cat(corrects), torch.cat(qts)
    overall = ece_confidence(C, OK)
    print(f"  {tag:<12} ECE {overall:.4f}   "
          f"ECE-dist {ece(P, T, M):.4f}   Brier {brier(P, T, M):.4f}   "
          f"NLL {nll(P, T, M):.4f}")
    for i, name in enumerate(TYPES):
        sel = QT == i
        if sel.any():
            print(f"      {name:<8} ECE {ece_confidence(C[sel], OK[sel]):.4f}"
                  f"   mean conf {C[sel].mean():.3f}   acc {OK[sel].float().mean():.3f}")
    return overall


if __name__ == "__main__":
    p = argparse.ArgumentParser()
    p.add_argument("--ckpt", default="jevlite.pt")
    p.add_argument("--bs", type=int, default=8)
    p.add_argument("--limit", type=int, default=None)
    p.add_argument("--config", default="all",
                   help="one workflow (customer_service, invoice_processing, "
                        "security_incidents, agent_trace_observability) instead "
                        "of all four - a domain specialist is where a small model wins")
    a = p.parse_args()

    ck = require_ckpt(a.ckpt)
    dev = "cuda" if torch.cuda.is_available() else "cpu"
    model = JevLite(ck["encoder"])
    model.load_state_dict(ck["state_dict"])
    model.to(dev).eval()

    _, va = val_split(load_split("train", a.config, limit=a.limit))
    ds = TypedDecisions(va, model.tok, ck["max_len"], model.qid, model.lid)
    dl = DataLoader(ds, batch_size=a.bs,
                    collate_fn=lambda b: collate(b, model.tok.pad_token_id))
    cached = collect(model, dl, dev)

    print(f"fitting on {len(va)} validation cases\n")
    ece_before = report(cached, torch.zeros(len(TYPES)), "before")
    log_temp = fit(cached)
    print()
    ece_after = report(cached, log_temp, "after")

    # Measured on the first full run: training against the soft consensus
    # distribution already calibrates the model, the fitted temperatures came
    # back at ~1.0, and applying them made ECE slightly WORSE. So this step is
    # now a selection, not an assumption - it keeps T=1 unless it earns its place.
    if ece_after >= ece_before:
        print(f"\n  temperature scaling did not help "
              f"({ece_before:.4f} -> {ece_after:.4f}) - keeping T=1.0.")
        print("  That is the expected outcome when the loss already trains on the")
        print("  consensus distribution; it means your calibration came for free.")
        log_temp = torch.zeros(len(TYPES))
    else:
        print(f"\n  temperature scaling helped: ECE {ece_before:.4f} -> "
              f"{ece_after:.4f}")

    print("\n  fitted temperatures:  " + "  ".join(
        f"{n}={log_temp.exp()[i]:.3f}" for i, n in enumerate(TYPES)))
    print("  (T > 1 means the model was overconfident on that type, T < 1 under)")

    model.log_temp.data = log_temp
    ck["state_dict"] = model.state_dict()
    ck["temperatures"] = log_temp.exp().tolist()
    torch.save(ck, a.ckpt)
    print(f"\nsaved -> {a.ckpt}")
'''

FILES['04_eval.py'] = r'''
"""Step 4 - score on the official 400-case test split and compare to Jev.

Reports the same headline metric Jev is published against (argmax accuracy over
all decisions), plus the things accuracy hides: calibration, distributional
agreement, the confidence/coverage tradeoff, and latency.
"""
import argparse
import json
import time
import torch
from torch.utils.data import DataLoader
from td_data import load_split, TypedDecisions, collate, pad_cat
from model import JevLite, argmax_per_question, require_ckpt
from typed_schema import TYPES
from metrics import (ece, ece_confidence, brier, nll, tvd, risk_coverage,
                     reliability_bins)

JEV_ACC = 0.727          # TypeSafe Jev 1.13.0, published on this test split
JEV_ECE_REPORTED = 0.144  # from the LocalLLaMA typed-decisions run


def evaluate(model, dl, dev, temp):
    P, T, M, TY, G = [], [], [], [], []
    pred_all, gold_all, conf_all, qtype_all = [], [], [], []
    n_cases = 0
    t0 = time.perf_counter()
    with torch.no_grad():
        for b in dl:
            b = {k: v.to(dev) for k, v in b.items()}
            _, probs = model(b, apply_temperature=temp)
            pred, conf = argmax_per_question(probs, b)
            gold, _ = argmax_per_question(b["target"], b)
            m = b["qmask"]
            pred_all.append(pred[m]); gold_all.append(gold[m])
            conf_all.append(conf[m]); qtype_all.append(b["qtype"][m])
            P.append(probs.cpu()); T.append(b["target"].cpu())
            M.append(b["label_mask"].cpu()); G.append(b["group"].cpu())
            TY.append(b["qtype"].clamp(min=0).gather(1, b["group"].clamp(min=0)).cpu())
            n_cases += b["input_ids"].size(0)
    dt = time.perf_counter() - t0

    cat = lambda xs: torch.cat([x.reshape(-1) if x.dim() == 1 else x for x in xs])
    pred, gold = cat(pred_all).cpu(), cat(gold_all).cpu()
    conf, qt = cat(conf_all).cpu(), cat(qtype_all).cpu()
    correct = pred == gold

    # Pad to the widest label dimension: batches spanning different workflows
    # do not share one, and a bare torch.cat silently only works when they do.
    Pm, Tm = pad_cat(P), pad_cat(T)
    Mm = pad_cat(M, False)
    TYm = pad_cat(TY)
    Gm = pad_cat(G, -1)
    return dict(correct=correct, conf=conf, qtype=qt, dt=dt, n_cases=n_cases,
                P=Pm, T=Tm, M=Mm, TY=TYm, G=Gm)


def show(r, tag):
    acc = r["correct"].float().mean().item()
    print(f"\n=== {tag} ===")
    print(f"  accuracy   {acc:.4f}   ({int(r['correct'].sum())}/{len(r['correct'])} decisions)")
    print(f"  ECE        {ece_confidence(r['conf'], r['correct']):.4f}"
          "   (top-1 confidence vs correctness - comparable to published Jev)")
    print(f"  ECE-dist   {ece(r['P'], r['T'], r['M']):.4f}"
          "   (predicted vs consensus distribution)")
    print(f"  Brier      {brier(r['P'], r['T'], r['M']):.4f}")
    print(f"  NLL        {nll(r['P'], r['T'], r['M']):.4f}")
    print(f"  TVD        {tvd(r['P'], r['T'], r['M'], r['G'], r['P'].size(1)):.4f}"
          "   (distance from the consensus distribution)")
    print("  per type:")
    for i, name in enumerate(TYPES):
        sel = r["qtype"] == i
        selm = r["M"] & (r["TY"] == i)
        if sel.any():
            print(f"    {name:<8} acc {r['correct'][sel].float().mean():.4f}"
                  f"   ECE {ece_confidence(r['conf'][sel], r['correct'][sel]):.4f}"
                  f"   n={int(sel.sum())}")
    return acc


if __name__ == "__main__":
    p = argparse.ArgumentParser()
    p.add_argument("--ckpt", default="jevlite.pt")
    p.add_argument("--bs", type=int, default=8)
    p.add_argument("--limit", type=int, default=None)
    p.add_argument("--config", default="all",
                   help="one workflow (customer_service, invoice_processing, "
                        "security_incidents, agent_trace_observability) instead "
                        "of all four - a domain specialist is where a small model wins")
    a = p.parse_args()

    ck = require_ckpt(a.ckpt)
    dev = "cuda" if torch.cuda.is_available() else "cpu"
    model = JevLite(ck["encoder"])
    model.load_state_dict(ck["state_dict"])
    model.to(dev).eval()

    test = load_split("test", a.config, limit=a.limit)
    ds = TypedDecisions(test, model.tok, ck["max_len"], model.qid, model.lid)
    dl = DataLoader(ds, batch_size=a.bs,
                    collate_fn=lambda b: collate(b, model.tok.pad_token_id))

    raw = evaluate(model, dl, dev, temp=False)
    show(raw, "uncalibrated")
    cal = evaluate(model, dl, dev, temp=True)
    acc = show(cal, "calibrated (per-type temperature)")

    print(f"\n=== vs Jev 1.13.0 ===")
    print(f"  JevLite  {acc:.4f}        Jev  {JEV_ACC:.4f}"
          f"   ->  {'AHEAD' if acc > JEV_ACC else 'behind'} by {abs(acc-JEV_ACC):.4f}")
    my_ece = ece_confidence(cal["conf"], cal["correct"])
    print(f"  JevLite ECE {my_ece:.4f}    Jev ECE {JEV_ECE_REPORTED:.4f} (reported)"
          f"   ->  {'AHEAD' if my_ece < JEV_ECE_REPORTED else 'behind'}")
    n_dec = len(cal["correct"])
    print(f"\n  {cal['dt']:.2f}s for {cal['n_cases']} cases / {n_dec} decisions"
          f"  ->  {cal['dt']/cal['n_cases']*1000:.1f} ms/case on {dev}, no network")
    print(f"  Jev API p50 measured from Europe: 239 ms/call")

    print("\n=== confidence -> coverage (can you autoroute?) ===")
    print(f"  {'threshold':>10}{'coverage':>11}{'accuracy':>11}")
    for t, cov, ac in risk_coverage(cal["conf"], cal["correct"]):
        print(f"  {t:>10.2f}{cov:>11.1%}{ac:>11.4f}")
    print("\n  Read the last column: if accuracy climbs sharply with the threshold,")
    print("  the confidence is real and you can route the top band automatically.")

    json.dump({
        "accuracy": acc,
        "ece": my_ece,
        "ece_dist": ece(cal["P"], cal["T"], cal["M"]),
        "brier": brier(cal["P"], cal["T"], cal["M"]),
        "nll": nll(cal["P"], cal["T"], cal["M"]),
        "tvd": tvd(cal["P"], cal["T"], cal["M"], cal["G"], cal["P"].size(1)),
        "ms_per_case": cal["dt"] / cal["n_cases"] * 1000,
        "n_cases": cal["n_cases"], "n_decisions": n_dec, "device": dev,
        "config": a.config,
        "per_type": [
            {"type": name,
             "accuracy": float(cal["correct"][cal["qtype"] == i].float().mean()),
             "ece": ece_confidence(cal["conf"][cal["qtype"] == i],
                                   cal["correct"][cal["qtype"] == i]),
             "n": int((cal["qtype"] == i).sum())}
            for i, name in enumerate(TYPES) if (cal["qtype"] == i).any()],
        "coverage": risk_coverage(cal["conf"], cal["correct"]),
        "reliability_raw": reliability_bins(raw["conf"], raw["correct"]),
        "reliability_cal": reliability_bins(cal["conf"], cal["correct"]),
    }, open("results.json", "w"), indent=1)
    print("\nwrote results.json  ->  python plots.py --show")
'''

FILES['05_serve.py'] = r'''
"""Step 5 - the Jev-shaped API, running locally for free.

  python 05_serve.py --demo
  python 05_serve.py --serve      # POST /decide {"state": {...}, "questions": {...}}

The questions are supplied per request. Nothing about the schema is baked into
the weights, so you can add a question, rename a label, or point it at a new
workflow without retraining - the same property that makes Jev usable as a
general decision endpoint.
"""
import argparse
import json
import torch
from td_data import encode, collate
from model import JevLite, require_ckpt
from typed_schema import iter_labels, Question

_engine = {}


def load(ckpt="jevlite.pt", device=None):
    ck = require_ckpt(ckpt)
    dev = device or ("cuda" if torch.cuda.is_available() else "cpu")
    m = JevLite(ck["encoder"])
    m.load_state_dict(ck["state_dict"])
    m.to(dev).eval()
    _engine.update(model=m, dev=dev, max_len=ck["max_len"],
                   temps=ck.get("temperatures"))
    return _engine


@torch.no_grad()
def decide(state, questions, calibrated=True):
    """-> {question: {"label": str, "confidence": float, "probabilities": {...}}}"""
    e = _engine
    m = e["model"]
    row = {"id": "live", "state": state, "questions": questions, "gold": {}}
    enc = encode(row, m.tok, e["max_len"], m.qid, m.lid, with_gold=False)
    b = collate([enc], m.tok.pad_token_id)
    b = {k: v.to(e["dev"]) for k, v in b.items()}
    _, probs = m(b, apply_temperature=calibrated)
    probs = probs[0].cpu()

    out, i = {}, 0
    for gi, qname in enumerate(enc["qnames"]):
        labels = enc["labels"][gi]
        vals = probs[i:i + len(labels)].tolist()
        i += len(labels)
        dist = dict(zip(labels, (round(v, 4) for v in vals)))
        top = max(dist, key=dist.get)
        out[qname] = {"label": top, "confidence": dist[top], "probabilities": dist}
    return out


DEMO_STATE = {
    "agent": {"autonomy": "checkpointed", "model": "internal-agent-v4"},
    "constraints": ["Do not touch customer data outside the named accounts"],
    "task": "Delete personal data for the accounts in the erasure queue.",
    "trace_summary": {"duration_s": 128.5, "steps": 7, "tool_errors": 1,
                      "irreversible_actions": 2, "constraint_violations": 1},
}

DEMO_QUESTIONS = {
    "action": Question.choice(
        "What should the observability system do with this trace?",
        {"continue": "Let the agent proceed without interruption.",
         "human_review": "Queue this trace for a human to review.",
         "observe": "Keep running, but flag the trace for later sampling.",
         "stop": "Halt the agent now."}),
    "needs_review": Question.noul(
        "This trace requires human review.",
        true_desc="A human should inspect this run.",
        false_desc="No human attention is warranted."),
    "risk": Question.score(
        "How risky was the agent's behaviour in this trace?",
        ["Benign: read-only or clearly safe actions.",
         "Low: routine writes within scope.",
         "Moderate: irreversible or out-of-scope actions.",
         "High: destructive, security-relevant, or policy-violating actions."]),
}

if __name__ == "__main__":
    p = argparse.ArgumentParser()
    p.add_argument("--ckpt", default="jevlite.pt")
    p.add_argument("--demo", action="store_true")
    p.add_argument("--serve", action="store_true")
    p.add_argument("--port", type=int, default=8000)
    a = p.parse_args()
    e = load(a.ckpt)
    if e["temps"]:
        print(f"loaded on {e['dev']}  temperatures={[round(t,3) for t in e['temps']]}")

    if a.serve:
        from fastapi import FastAPI
        from pydantic import BaseModel
        from typing import Any
        import uvicorn

        class Req(BaseModel):
            state: Any
            questions: dict
            calibrated: bool = True

        app = FastAPI(title="JevLite")

        @app.post("/decide")
        def _decide(r: Req):
            return decide(r.state, r.questions, r.calibrated)

        uvicorn.run(app, host="0.0.0.0", port=a.port)
    else:
        import time
        t0 = time.perf_counter()
        out = decide(DEMO_STATE, DEMO_QUESTIONS)
        print(json.dumps(out, indent=2))
        print(f"\n{(time.perf_counter()-t0)*1000:.1f} ms, 3 typed questions, "
              "one forward pass, $0")
'''

FILES['06_export_onnx.py'] = r'''
"""Step 6 - export to ONNX so it runs anywhere for free.

ONNX Runtime Web gives you the same model in a browser tab over WebGPU, with
WASM as the fallback. That is the one advantage an API model structurally cannot
match: no network hop, no key, no per-call cost, and the data never leaves the
machine. Measured reference for a 150M encoder at this shape is ~35 ms p50 on
single-thread WASM, and less on WebGPU.

  python 06_export_onnx.py --quantize
"""
import argparse
import os
import torch
from td_data import load_split, TypedDecisions, collate
from model import JevLite, require_ckpt


class ExportWrapper(torch.nn.Module):
    """Raw per-label logits only. The grouped softmax is a dozen lines of JS and
    keeping it out of the graph avoids exporting a Python loop over questions."""

    def __init__(self, m):
        super().__init__()
        self.m = m

    def forward(self, input_ids, attention_mask, label_pos):
        return self.m.logits(input_ids, attention_mask, label_pos)


if __name__ == "__main__":
    p = argparse.ArgumentParser()
    p.add_argument("--ckpt", default="jevlite.pt")
    p.add_argument("--out", default="jevlite.onnx")
    p.add_argument("--opset", type=int, default=18,
                   help="18 is the floor: torch emits opset-18 ops "
                        "(Split.num_outputs) and a lower request "
                        "silently produces a graph ORT refuses to load")
    p.add_argument("--quantize", action="store_true", help="int8 dynamic, ~4x smaller")
    a = p.parse_args()

    ck = require_ckpt(a.ckpt)
    model = JevLite(ck["encoder"])
    model.load_state_dict(ck["state_dict"])
    model.eval()

    row = load_split("test", limit=1)[0]
    ds = TypedDecisions([row], model.tok, ck["max_len"], model.qid, model.lid)
    b = collate([ds[0]], model.tok.pad_token_id)

    torch.onnx.export(
        ExportWrapper(model),
        (b["input_ids"], b["attention_mask"], b["label_pos"]),
        a.out,
        input_names=["input_ids", "attention_mask", "label_pos"],
        output_names=["label_logits"],
        dynamic_axes={"input_ids": {0: "batch", 1: "seq"},
                      "attention_mask": {0: "batch", 1: "seq"},
                      "label_pos": {0: "batch", 1: "labels"},
                      "label_logits": {0: "batch", 1: "labels"}},
        opset_version=a.opset,
    )
    sidecar = a.out + ".data"
    extra = f" + {os.path.getsize(sidecar)/1e6:.0f} MB {sidecar}" if \
        os.path.exists(sidecar) else ""
    print(f"exported -> {a.out} ({os.path.getsize(a.out)/1e6:.1f} MB){extra}")
    if extra:
        print("  note: weights live in the sidecar. Serve BOTH files from the same")
        print("  directory, or ORT Web will load a graph with no weights in it.")

    # Parity check. An export that silently diverges is worse than no export -
    # you would only find out from wrong decisions in production.
    import numpy as np
    import onnxruntime as ort
    with torch.no_grad():
        ref = ExportWrapper(model)(b["input_ids"], b["attention_mask"],
                                   b["label_pos"]).numpy()
    sess = ort.InferenceSession(a.out, providers=["CPUExecutionProvider"])
    got = sess.run(None, {k: b[k].numpy() for k in
                          ("input_ids", "attention_mask", "label_pos")})[0]
    diff = float(np.abs(ref - got).max())
    print(f"  parity vs pytorch: max abs diff {diff:.2e} "
          f"{'OK' if diff < 1e-3 else 'FAILED - do not ship this'}")

    if a.quantize:
        from onnxruntime.quantization import quantize_dynamic, QuantType
        q = a.out.replace(".onnx", ".int8.onnx")
        quantize_dynamic(a.out, q, weight_type=QuantType.QUInt8)
        qsize = os.path.getsize(q) + (os.path.getsize(q + ".data")
                                      if os.path.exists(q + ".data") else 0)
        print(f"quantized -> {q} ({qsize/1e6:.0f} MB total)")
        qs = ort.InferenceSession(q, providers=["CPUExecutionProvider"])
        qgot = qs.run(None, {k: b[k].numpy() for k in
                             ("input_ids", "attention_mask", "label_pos")})[0]
        print(f"  int8 vs fp32: max abs diff {np.abs(got - qgot).max():.2e}")
        print("  int8 shifts logits. Re-run 03_calibrate.py against the quantized")
        print("  model before trusting its confidences.")

    temps = ck.get("temperatures")
    print("\nIn the browser, load this with onnxruntime-web, tokenize with")
    print("@huggingface/transformers, then for each question softmax its own")
    print(f"label logits after dividing by its type temperature: {temps}")
'''

for name, text in FILES.items():
    pathlib.Path(name).write_text(text.lstrip('\n') + '\n', encoding='utf-8')
print(f'wrote {len(FILES)} files:', ', '.join(sorted(FILES)))

## 2. Smoke test

Checks that every label marker lands where the model expects it, that
grouped softmax normalises per question and stays zero on padding, and
that gold distributions key on exactly the labels we score.

`--all` runs that last check across **all 1,600 rows**, which is not
paranoia: 10% of this dataset is bare `noul` questions carrying no
`criteria` key at all, and that shape does not appear in the first rows
of the first workflow. A 6-row check passes while training dies 300
rows in. ~2 minutes, no GPU needed - cheaper than finding out later.

In [ ]:
run('python smoke_test.py --all')

## 3. What you have to beat

Free numbers, none of which need a trained model:

- **single annotator** - how often one random rater agrees with the
  consensus. A reference point, *not* a cap: a model that always picks
  the consensus argmax scores 1.000, and Jev already exceeds this.
- **ambiguous** - share of decisions that are near-ties. This is the
  real limit on headline accuracy; above it you are coin-flipping.
- **majority** - always answer the most common label.
- **frozen** - frozen encoder embeddings + logistic regression. This is
  the baseline that beat Jev by 10 points on Banking77 at 44x the speed.
  If the fine-tune below cannot clear it, ship this instead.

In [ ]:
run('python 01_ceiling.py')

## 4. Train

Trains against the consensus **distribution** (soft cross-entropy +
Brier), not the argmax. Training on the winner is the direct cause of
overconfidence: a model taught that a 55/45 case is `true` learns to say
0.95 and is then wrong 45% of the time while claiming near-certainty.

20 epochs is ~25 min on a T4, and it stops early once validation
stalls for 6 epochs - the best checkpoint is kept either way, so a
longer budget cannot make the result worse, only the wait longer.

Do **not** lower `--max-len`: states run 401-997 tokens (p50 588), so
512 would truncate 73% of them. Batches already pad to their own
longest example, so the high cap costs nothing.

In [ ]:
run('python 02_train.py --epochs 20 --bs 4 --accum 4 --max-len 1024')

## 5. Calibrate

One temperature **per primitive type**, fitted on a held-out slice of
train. This targets Jev's weakest measured claim: its miscalibration
runs in opposite directions by type (`choice`/`score` overconfident,
`noul` underconfident), so a single global temperature cannot fix both.

**Measured result from the first full run: this step did nothing.**
The fitted temperatures came back at [1.09, 1.00, 0.98] and applying
them made ECE slightly worse (0.0653 -> 0.0693). Training against the
soft consensus distribution had already done the calibration work.

So the step now *selects* rather than assumes: it keeps T=1.0 unless
the fit measurably improves validation ECE. A no-op here is a good
outcome - it means your calibration came free with the loss function.
Seconds, on CPU.

In [ ]:
run('python 03_calibrate.py')

## 6. Score against Jev

The official 400-case test split, 2,000 decisions. Watch three things:

- **accuracy** vs Jev's 0.727 - but ~16% of this split is near-ties,
  so treat small differences as noise
- **ECE** - the top-1-confidence-vs-correctness one, which is what
  published Jev numbers measure. `ECE-dist` answers a different question
  and the two can disagree sharply, so never quote one alone.
- **confidence -> coverage** - if accuracy climbs steeply with the
  threshold, the confidence is real and you can autoroute the top band.

In [ ]:
run('python 04_eval.py')

## 7. The charts

Every step above dumps its numbers to JSON, and this reads them back.
Five figures, each answering one question:

1. **Scoreboard** - where you landed against Jev and against the
   baselines that need no training at all.
2. **Training curve** - loss and validation accuracy per epoch, in
   separate panels (two scales never share one axis).
3. **Reliability** - confidence against how often the model was
   actually right, before and after temperature scaling. Below the
   diagonal is overconfident; the gap *is* the ECE.
4. **Coverage** - accuracy on the decisions you keep as you raise the
   confidence threshold. This is the chart that says whether you can
   autoroute the confident band and escalate the rest.
5. **Per type** - accuracy and calibration split by noul/choice/score,
   because Jev's own weakness is type-dependent.

Figures also land in `plots/` as PNGs you can drop into a write-up.

In [ ]:
run('python plots.py --outdir plots')

from IPython.display import Image, display
import glob
for f in sorted(glob.glob('plots/*.png')):
    display(Image(filename=f))

## 8. Dynamic schema

The point of the label-embedding head: these questions were never seen
in this combination during training, and no retraining happened. The
schema is data at inference time.

In [ ]:
run('python 05_serve.py --demo')

## 8b. Want more accuracy? Go bigger (optional, ~45 min)

The base encoder is 150M parameters. The other open reproduction of
this architecture used DeBERTa-v3-large (0.4B) and reported 0.854
in-domain, so capacity is the next lever after epochs.

ModernBERT-large is ~395M and still fits a free T4 at batch 2 with
8-step accumulation (same effective batch of 16). It roughly triples
the epoch time, so this is a coffee-length run, not a quick one.

It writes to a separate checkpoint, so your base model survives. To
score it, rerun the eval with `--ckpt jevlite-large.pt`.

**If it OOMs**, drop to `--bs 1 --accum 16` before anything else -
the effective batch stays 16 and only the step count changes.

In [ ]:
# Uncomment to run. Check the charts from the base model first:
# if its validation curve was still climbing, more epochs is the
# cheaper win and you should spend the session on that instead.

# run('python 02_train.py --encoder answerdotai/ModernBERT-large '
#     '--epochs 20 --bs 2 --accum 8 --max-len 1024 '
#     '--out jevlite-large.pt')
# run('python 03_calibrate.py --ckpt jevlite-large.pt')
# run('python 04_eval.py --ckpt jevlite-large.pt')
# run('python plots.py --outdir plots-large')

## 9. Keep the checkpoint

Colab reclaims sessions with ~90 minutes of idle time and you lose the
filesystem with it. Save `jevlite.pt` somewhere before you close the tab.

Drive is the zero-setup option; the Hub cell below it is better if you
want the model reachable from anywhere.

In [ ]:
# Optional and non-fatal. The mount needs you to authorise a popup,
# so it cannot run unattended and will raise if you skip it or have
# no Drive. Nothing below depends on this cell succeeding.
import os
import shutil

try:
    from google.colab import drive
    drive.mount('/content/drive')
    dest = '/content/drive/MyDrive/jevlite'
    os.makedirs(dest, exist_ok=True)
    shutil.copy('jevlite.pt', dest)
    print('saved to', dest)
except Exception as e:
    print(f'Drive save skipped ({type(e).__name__}: {e}).')
    print('Use the download cell or the Hub cell below instead - or',
          'just rerun the notebook later, it is only ~20 minutes.')

In [ ]:
# Simplest alternative: download the checkpoint straight to your machine.
from google.colab import files
files.download('jevlite.pt')

In [ ]:
# Optional: push to the Hugging Face Hub instead (needs a free write token)
# from huggingface_hub import notebook_login, HfApi
# notebook_login()
# HfApi().upload_file(path_or_fileobj='jevlite.pt',
#                     path_in_repo='jevlite.pt',
#                     repo_id='YOUR_USERNAME/jevlite', repo_type='model')

## 10. Export for local / in-browser inference

This is the advantage an API structurally cannot match: no network hop,
no key, no per-call cost, and the data never leaves the machine. Jev's
measured p50 from Europe is 239 ms; a 150M encoder at this shape runs
~35 ms p50 on single-thread WASM and less on WebGPU.

Two things to know before you ship the result:

- the weights land in a ~596 MB `jevlite.onnx.data` sidecar - serve it
  alongside the graph or you load a model with no weights
- int8 shrinks it to ~151 MB but moves logits materially, so rerun
  step 5 against the quantized model rather than reusing its temperatures

The parity check in this cell is the one that matters: an export that
silently diverges from PyTorch only shows up as wrong decisions later.

In [ ]:
%pip install -q onnx onnxruntime onnxscript
run('python 06_export_onnx.py --quantize')

In [ ]:
from google.colab import files
files.download('jevlite.onnx')
files.download('jevlite.onnx.data')

## Where to go next

- **Beat it on your own data.** The independent Banking77 result says a
  small model that has seen your workflow beats a general model that has
  not. Point `td_data.load_split` at your own states and questions.
- **Try a bigger encoder.** `--encoder microsoft/deberta-v3-large`
  (0.4B) if the T4 has room - then check the gain is not confined to the
  ambiguous near-tie decisions, where it would be luck.
- **Do not chase the last accuracy point.** ~16% of this split is
  near-ties. Gains there are luck; gains in the coverage table are real.